In [1]:
# %pip install torch torchvision pandas

#### **Notebook-Zelle 1 – Bibliotheken und MNIST laden**

In [2]:
import time             #Trainingszeit messen
import torch            # Neuronales Netzwerk erstellen
import torch.nn as nn   # 	Netzwerkschichten und Funktionen
import torch.optim as optim   # Optimierung beim Lernen
import pandas as pd         #CSV-Dateien einlesen
import numpy as np         # Mathematische Berechnungen

import math

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader



In [3]:
# Parameter
LEARNING_RATE = 0.05
EPOCHS = 20 # Number of training epochs
HIDDEN_NODES = 1000 
BATCH_SIZE = 64
SEED = 42 # Startwert für Zufallszahlen

In [4]:
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

#### **1. CSV-Daten laden**

In [5]:

data = pd.read_csv(
    "./datasets/mnist_train_100.csv",
    header=None #sagt pandas, dass die erste Zeile Daten und keine Überschrift enthält.
    )
print(data.shape)
display(data.head())
#print(data.head())
print("Anzahl der Zeilen:", len(data))

# Alle Spaltennamen als Liste
print(data.columns.tolist())
print(data.columns)

# Informationen zu Spalten und Datentypen
data.info()


(100, 785)


,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Anzahl der Zeilen: 100
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 2

**Ergebnis**: Die Datei enthält 785 Spalten:
- Erste Spalte: richtige Ziffer (Label).
- Weitere 784 Spalten: Pixelwerte.

Das bedeutet:
- Du hast aktuell nur **10 Bilder** in deiner Datei.
- Jedes Bild hat **784 Pixelwerte** und zusätzlich ein Label.
- Die Datei ist also grundsätzlich richtig aufgebaut – aber beim Erstellen von X_train und X_test wurden die Daten offenbar falsch ausgewählt

#### **2. Daten aufteilen**

Schauen wir uns die einzelnen Teile an:
- data ist deine Tabelle mit den MNIST-Daten.
- **.iloc** wählt Daten anhand ihrer Position aus.
-  **:** bedeutet alle Zeilen.
- **0** bedeutet die erste Spalte. Python fängt beim Zählen bei 0 
an.
- **1:** – nimmt alle Spalten ab Position 1, also ab der zweiten Spalte, bis zum Ende.
- labels = speichert das Ergebnis unter dem Namen labels
- images = Hier bedeutet 1: Nimm alle Spalten ab der zweiten Spalte bis zum Ende.

In [6]:
labels = data.iloc[:, 0] # Nimm alle Zeilen aus der ersten Spalte von data und speichere sie in der Variable labels.
images = data.iloc[:, 1:] # Nimm alle Zeilen und alle Spalten ab der zweiten
#Spalte bis zur letzten Spalte aus dem DataFrame data.

print("Labels:", labels.shape)
print("Pixelwerte:", images.shape)

Labels: (100,)
Pixelwerte: (100, 784)


- **Labels** (10,): Du hast 10 Bildbeschriftungen (also die jeweils dargestellte Ziffer).
- **Pixelwerte** (10, 784): Du hast 10 Bilder mit jeweils 784 Pixelwerten.

In [7]:

X = images
# Ziffern als Zielwerte
y = labels

# 80 % Training, 20 % Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Trainingsdaten:", X_train.shape)
print("Testdaten:", X_test.shape)

Trainingsdaten: (80, 784)
Testdaten: (20, 784)


**Was passiert hier?**
- X enthält die Pixelwerte.
- y enthält die richtigen Ziffern.
- 80 % der Bilder werden zum Trainieren verwendet.
- 20 % werden zum Testen verwendet.

#### **4. Daten in PyTorch-Tensoren umwandeln**
PyTorch benötigt Tensoren, um mit den Daten rechnen zu können.
Die Pixelwerte werden durch 255.0 geteilt, damit sie zwischen 0 und 1 liegen.

In [8]:


# Pixelwerte in Tensoren umwandeln
X_train_tensor = torch.tensor(
    X_train.to_numpy(), dtype=torch.float32
) / 255.0

X_test_tensor = torch.tensor(
    X_test.to_numpy(), dtype=torch.float32
) / 255.0

# Richtige Ziffern in Tensoren umwandeln
y_train_tensor = torch.tensor(
    y_train.to_numpy(), dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test.to_numpy(), dtype=torch.long
)

# PyTorch-Datensätze erstellen
train_data = TensorDataset(X_train_tensor, y_train_tensor)
test_data = TensorDataset(X_test_tensor, y_test_tensor)

In [9]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("X_train_tensor:", X_train_tensor.shape)
print("X_test_tensor:", X_test_tensor.shape)

print("Trainingsdaten:", len(train_data))

X_train: (80, 784)
X_test: (20, 784)
X_train_tensor: torch.Size([80, 784])
X_test_tensor: torch.Size([20, 784])
Trainingsdaten: 80


#### **5. PyTorch-Datensätze erstellen**

In [10]:
train_data = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_data = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

In [11]:
train_loader = DataLoader(
    train_data,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=2,
    shuffle=False
)

In [12]:
images, labels = next(iter(train_loader))

print("Bilder im Batch:", images.shape)
print("Labels im Batch:", labels.shape)

Bilder im Batch: torch.Size([8, 784])
Labels im Batch: torch.Size([8])


Das bedeutet:
- **8 Bilder**: Dein Modell verarbeitet 8 Bilder gleichzeitig.
- **784 Pixelwerte**: Jedes Bild hat 784 Pixelwerte (28 × 28 Pixel).
- **8 Labels**: Zu jedem Bild gehört eine richtige Ziffer als Beschriftung.

In [13]:
images, labels = next(iter(test_loader))

print("Bilder im Batch:", images.shape)
print("Labels im Batch:", labels.shape)

Bilder im Batch: torch.Size([2, 784])
Labels im Batch: torch.Size([2])


#### **6. Parameter festlegen**

In [14]:
LEARNING_RATE = 0.05
EPOCHS = 20
HIDDEN_NODES = 1000
BATCH_SIZE = 64
SEED = 42

torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Verwendetes Gerät:", device)

Verwendetes Gerät: cpu


#### **7. Neuronales Netzwerk erstellen**


<span style="color: blue;">Eingabeschicht:</span>
784 Neuronen (Die 784 Pixelwerte eines Bildes)
<p   style="color: blue">
  &darr;
</p>
<span style="color: blue;">Versteckte Schicht:</span>
1000 Neuronen (Lernt Muster und Merkmale der Ziffern)
<p   style="color: blue">
  &darr;
</p>
<span style="color: blue;">Ausgabeschicht:</span>
10 Neuronen (Mögliche Ziffern von 0 bis 9)

In [15]:
class MNIST_FNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(784, HIDDEN_NODES)
        self.relu = nn.ReLU()
        self.output = nn.Linear(HIDDEN_NODES, 10)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1)
        x = self.relu(self.hidden(x))
        return self.output(x)

In [16]:
fc1 = nn.Linear(784, 1000)

#### **8. Gewichtinitialisierung programmieren**

Methode	      
- uniform_1:	Gewichte zufällig zwischen -1 und 1
- uniform_scaled:	Gewichte in einem kleineren Wertebereich
- normal:	Gewichte aus einer Normalverteilung

In [17]:
def initialize_weights(model, method):

    for layer in model.modules():

        if isinstance(layer, nn.Linear):

            if method == "uniform_1":
                nn.init.uniform_(layer.weight, -1, 1)

            elif method == "uniform_scaled":
                limit = 1 / np.sqrt(HIDDEN_NODES)
                nn.init.uniform_(
                    layer.weight,
                    -limit,
                    limit
                )

            elif method == "normal":
                nn.init.normal_(
                    layer.weight,
                    mean=0.0,
                    std=0.5
                )

            nn.init.zeros_(layer.bias)

In [18]:

def experiment(method):

    # 1. Zufallsstartwert festlegen
    torch.manual_seed(SEED)

    # 2. Neuronales Netzwerk erstellen
    model = MNIST_FNN().to(device)

    # 3. Gewichte initialisieren
    initialize_weights(model, method)

    # 4. Generator für reproduzierbare Datenmischung
    generator = torch.Generator().manual_seed(SEED)

    # 5. DataLoader für Trainingsdaten
    train_loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0
    )

    # 6. DataLoader für Testdaten
    test_loader = DataLoader(
        test_data,
        batch_size=1000,
        shuffle=False,
        num_workers=0
    )

    # 7. Verlustfunktion und Optimierer
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE
    )

    # 8. Training starten
    start = time.perf_counter()

    for epoch in range(EPOCHS):

        model.train()
        loss_sum = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            loss_sum += loss.item()

        print(
            f"{method} | Epoche {epoch + 1}/{EPOCHS}"
            f" | Loss: {loss_sum / len(train_loader):.4f}"
        )

    if device.type == "cuda":
        torch.cuda.synchronize()

    train_time = time.perf_counter() - start

    # 9. Netzwerk testen
    model.eval()

    correct = 0
    total = 0

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            predictions = model(images).argmax(dim=1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()

    test_time = time.perf_counter() - start

    # 10. Erfolgsrate berechnen
    accuracy = 100 * correct / total

    # 11. Ergebnisse zurückgeben
    return {
        "Initialisierung": method,
        "Erfolgsrate (%)": round(accuracy, 4),
        "Training (s)": round(train_time, 2),
        "Test (s)": round(test_time, 2)
    }


In [19]:
results = []

methods = [
    "uniform_1",
    "uniform_scaled",
    "normal"
]

for method in methods:

    result = experiment(method)

    results.append(result)

    print(result)

uniform_1 | Epoche 1/20 | Loss: 122.4617
uniform_1 | Epoche 2/20 | Loss: 70.3742
uniform_1 | Epoche 3/20 | Loss: 64.3159
uniform_1 | Epoche 4/20 | Loss: 21.2122
uniform_1 | Epoche 5/20 | Loss: 5.5880
uniform_1 | Epoche 6/20 | Loss: 2.4374
uniform_1 | Epoche 7/20 | Loss: 2.5079
uniform_1 | Epoche 8/20 | Loss: 1.5968
uniform_1 | Epoche 9/20 | Loss: 3.9530
uniform_1 | Epoche 10/20 | Loss: 0.1702
uniform_1 | Epoche 11/20 | Loss: 0.0003
uniform_1 | Epoche 12/20 | Loss: 0.0001
uniform_1 | Epoche 13/20 | Loss: 0.0001
uniform_1 | Epoche 14/20 | Loss: 0.0000
uniform_1 | Epoche 15/20 | Loss: 0.0001
uniform_1 | Epoche 16/20 | Loss: 0.0001
uniform_1 | Epoche 17/20 | Loss: 0.0000
uniform_1 | Epoche 18/20 | Loss: 0.0000
uniform_1 | Epoche 19/20 | Loss: 0.0000
uniform_1 | Epoche 20/20 | Loss: 0.0000
{'Initialisierung': 'uniform_1', 'Erfolgsrate (%)': 55.0, 'Training (s)': 0.46, 'Test (s)': 0.01}
uniform_scaled | Epoche 1/20 | Loss: 2.3107
uniform_scaled | Epoche 2/20 | Loss: 2.2553
uniform_scaled | E

In [20]:
results_df = pd.DataFrame(results)

display(results_df)

,Initialisierung,Erfolgsrate (%),Training (s),Test (s)
0,uniform_1,55.0,0.46,0.01
1,uniform_scaled,45.0,0.27,0.00
2,normal,50.0,0.27,0.00


- **Erfolgsrate** (%) bedeutet, wie viele Bilder das Netzwerk richtig erkannt hat.

- **Training (s)** bedeutet, wie lange das Training gedauert hat.

- **Test (s)** bedeutet, wie lange das Testen gedauert hat.

Du erhältst eine Tabelle mit:
- Initialisierungsmethode
- Erfolgsrate in Prozent
- Trainingszeit in Sekunden
- Testzeit in Sekunde

### Interpretation der Ergebnisse  mit 10 Daten

Während des Trainings sinkt der Loss bei den verschiedenen Initialisierungsmethoden teilweise sehr stark. Das zeigt, dass das neuronale Netz die Trainingsdaten lernt.

Allerdings beträgt die Erfolgsrate bei den gezeigten Ergebnissen 0 %. Das bedeutet, dass keine der Testvorhersagen richtig war.

Ein möglicher Grund dafür ist die sehr kleine Datenmenge. Der Datensatz enthält nur 10 Bilder, davon werden 8 zum Trainieren und 2 zum Testen verwendet. Dadurch kann das Netzwerk die Trainingsbilder zwar lernen, aber keine zuverlässige Erkennung neuer Bilder erreichen.

Die Ergebnisse sind deshalb nur eingeschränkt aussagekräftig. Um die Initialisierungsmethoden zuverlässig vergleichen zu können, wäre ein deutlich größerer Datensatz notwendig.


### Auswertung der Initialisierungsmethoden

Die drei Initialisierungsmethoden wurden hinsichtlich ihrer Erfolgsrate und Trainingszeit miteinander verglichen.

Die Methode `uniform_1` erreichte mit 55 % die höchste Erfolgsrate, benötigte jedoch mit 0,48 Sekunden die längste Trainingszeit.

Die Methode `uniform_scaled` erreichte eine Erfolgsrate von 45 % und benötigte 0,32 Sekunden. Die Methode `normal` erreichte 50 % und hatte mit 0,31 Sekunden die kürzeste Trainingszeit.

In diesem Versuch lieferte `uniform_1` somit die beste Erkennungsleistung. Die Methode `normal` war hingegen am schnellsten.

Die Ergebnisse sind jedoch nur eingeschränkt aussagekräftig, da der verwendete Datensatz mit nur 10 Bildern sehr klein ist. Außerdem können die Ergebnisse durch die zufällige Initialisierung und die Aufteilung der Trainings- und Testdaten beeinflusst werden.





<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Initialisierung</th>
      <th>Erfolgsrate (%)</th>
      <th>Training (s)</th>
      <th>Test (s)</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>uniform_1</td>
      <td>55.0</td>
      <td>0.34</td>
      <td>0.0</td>
    </tr>
    <tr>
      <th>1</th>
      <td>uniform_scaled</td>
      <td>45.0</td>
      <td>0.14</td>
      <td>0.0</td>
    </tr>
    <tr>
      <th>2</th>
      <td>normal</td>
      <td>50.0</td>
      <td>0.19</td>
      <td>0.0</td>
    </tr>
  </tbody>
</table>
</div>